# Практикум 7. Классификация текстов: Naive Bayes, SVM и анализ тональности

В лаборатории 7 вы видели, как каждое слово отзыва «голосует» за положительный или отрицательный класс, как сглаживание Лапласа спасает от нулевых вероятностей, как отрицание «не» меняет картину и как SVM ищет разделяющую полосу. Здесь повторим всё это в коде на 60 коротких отзывах:

1. напишем мультиномиальный наивный Байес с нуля (логарифмы вероятностей, сглаживание Лапласа) и сверим его с `sklearn.naive_bayes.MultinomialNB`;
2. разберём, какие слова тянут отзыв в какую сторону, и научим модель понимать отрицание («не понравился» → `НЕ_понравился`);
3. обучим линейный SVM градиентным спуском по hinge loss на `numpy`, затем `LinearSVC` и `LogisticRegression` из scikit-learn;
4. оценим модели: матрица ошибок, точность и полнота, кросс-валидация.

*Практикум к лаборатории 7 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

## 1. Данные: 60 отзывов

30 положительных и 30 отрицательных отзывов о фильмах, кафе, книгах и технике. Отзывы специально парные: одни и те же темы («актёры», «качество», «обслуживание») встречаются в обоих классах, поэтому модели придётся смотреть на оценочные слова. Последние 6 отзывов каждого класса отложим для теста: модель их не увидит при обучении.

*Совет: свои данные можно загрузить через панель «Файлы» слева (например, `reviews.csv` с колонками `text,label`) и прочитать `pd.read_csv("reviews.csv")`.*

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pos = [
    "Фильм очень понравился, отличный сюжет",
    "Отличный телефон, всем советую",
    "Еда вкусная, обслуживание отличное",
    "Книга интересная, советую прочитать",
    "Хороший ресторан, всё очень вкусно",
    "Наушники хорошие, звук отличный",
    "Совсем не скучно, фильм понравился",
    "Цена не высокая, качество хорошее",
    "Доставка быстрая, всё понравилось, спасибо",
    "Ни разу не пожалел, отличная покупка",
    "Персонал вежливый, номер чистый, рекомендую",
    "Очень доволен, телефон работает быстро",
    "Интересный сюжет, хорошие актёры",
    "Вкусный плов и приятная музыка",
    "Книга понравилась, читается легко",
    "Не разочаровал, рекомендую всем",
    "Отличное кафе, вкусно и недорого",
    "Камера хорошая, очень доволен покупкой",
    "Актёры играют отлично, советую",
    "Обслуживание быстрое и вежливое",
    "Удобный и красивый, рекомендую",
    "Хороший фильм для всей семьи",
    "Всё понравилось, придём ещё",
    "Качество отличное, спасибо продавцу",
    # тест
    "Отличный фильм, очень понравился",
    "Вкусно, быстро и недорого, советую",
    "Совсем не скучно, интересная книга",
    "Хорошее качество, не пожалел о покупке",
    "Персонал вежливый, всё понравилось",
    "Не разочаровал, отличный телефон",
]
neg = [
    "Фильм совсем не понравился, скучный сюжет",
    "Плохой телефон, никому не советую",
    "Еда холодная, обслуживание ужасное",
    "Книга скучная, не советую читать",
    "Плохой ресторан, всё невкусно",
    "Наушники плохие, звук ужасный",
    "Очень скучно, фильм не понравился",
    "Цена высокая, качество плохое",
    "Доставка медленная, ничего не понравилось",
    "Очень пожалел, ужасная покупка",
    "Персонал грубый, номер грязный, не рекомендую",
    "Очень разочарован, телефон сломался через неделю",
    "Скучный сюжет, плохие актёры",
    "Холодный плов и громкая музыка",
    "Книга не понравилась, читается тяжело",
    "Разочаровал, не рекомендую никому",
    "Ужасное кафе, невкусно и дорого",
    "Камера плохая, очень разочарован покупкой",
    "Актёры играют ужасно, не советую",
    "Обслуживание медленное и грубое",
    "Неудобный и тяжёлый, не рекомендую",
    "Плохой фильм, жаль потраченного времени",
    "Ничего не понравилось, больше не придём",
    "Качество ужасное, деньги не вернули",
    # тест
    "Ужасный фильм, совсем не понравился",
    "Невкусно, медленно и дорого, не советую",
    "Очень скучно, неинтересная книга",
    "Плохое качество, пожалел о покупке",
    "Персонал грубый, ничего не понравилось",
    "Разочаровал, плохой телефон",
]

def tokenize(text):
    return re.findall(r"\w+", text.lower())

# последние 6 отзывов каждого класса — тест
train = [(t, "+") for t in pos[:24]] + [(t, "-") for t in neg[:24]]
test = [(t, "+") for t in pos[24:]] + [(t, "-") for t in neg[24:]]
X_train, y_train = [t for t, _ in train], np.array([c for _, c in train])
X_test, y_test = [t for t, _ in test], np.array([c for _, c in test])
print(f"обучение: {len(train)} ({(y_train == '+').sum()} +, {(y_train == '-').sum()} −);  тест: {len(test)}")
lens = [len(tokenize(t)) for t in pos + neg]
print(f"слов в отзыве: от {min(lens)} до {max(lens)}, в среднем {np.mean(lens):.1f}")
print("\nПримеры:")
for t, c in train[:3] + train[24:27]:
    print(f"  {c}  {t}")

## 2. Мультиномиальный наивный Байес с нуля

Отзыв $d$ — мешок слов $w_1,\dots,w_n$. По формуле Байеса и «наивному» предположению о независимости слов

$$\hat c=\arg\max_c\Big[\log P(c)+\sum_{i=1}^{n}\log P(w_i\mid c)\Big],\qquad P(w\mid c)=\frac{\text{count}(w,c)+\alpha}{\sum_{v}\text{count}(v,c)+\alpha|V|}.$$

$\alpha=1$ — сглаживание Лапласа: без него слово, которого не было в классе, дало бы $P=0$ и $\log P=-\infty$, и один такой «ноль» перечеркнул бы весь отзыв. Логарифмы складываем вместо того, чтобы перемножать вероятности: произведение сотен маленьких чисел уходит в машинный ноль.

In [ ]:
from collections import Counter

def train_nb(docs, labels, alpha=1.0, featurize=tokenize):
    classes = sorted(set(labels))
    counts = {c: Counter() for c in classes}
    n_docs = Counter(labels)
    for d, c in zip(docs, labels):
        counts[c].update(featurize(d))
    vocab = sorted(set(w for c in classes for w in counts[c]))
    log_prior = {c: np.log(n_docs[c] / len(docs)) for c in classes}
    log_lik = {}
    for c in classes:
        total = sum(counts[c].values())
        log_lik[c] = {w: np.log((counts[c][w] + alpha) / (total + alpha * len(vocab))) for w in vocab}
    return dict(classes=classes, vocab=set(vocab), log_prior=log_prior, log_lik=log_lik,
                counts=counts, featurize=featurize)

def nb_scores(model, text):
    words = [w for w in model["featurize"](text) if w in model["vocab"]]   # незнакомые слова пропускаем
    return {c: model["log_prior"][c] + sum(model["log_lik"][c][w] for w in words) for c in model["classes"]}

def nb_predict(model, text):
    s = nb_scores(model, text)
    return max(s, key=s.get)

nb = train_nb(X_train, y_train)
print("словарь:", len(nb["vocab"]), "слов;  log P(+) =", round(nb["log_prior"]["+"], 3), " log P(−) =", round(nb["log_prior"]["-"], 3))
for c in nb["classes"]:
    print(f"  слов в классе {c}: {sum(nb['counts'][c].values())},  самые частые:", nb["counts"][c].most_common(6))

print("\nТаблица правдоподобий для нескольких слов (α = 1):")
rows = []
for w in ["не", "очень", "понравился", "скучно", "отличный", "советую", "плохой", "ужасное"]:
    rows.append([w, nb["counts"]["+"][w], nb["counts"]["-"][w], np.exp(nb["log_lik"]["+"][w]), np.exp(nb["log_lik"]["-"][w])])
print(pd.DataFrame(rows, columns=["слово", "count(+)", "count(−)", "P(w|+)", "P(w|−)"]).round(4).to_string(index=False))

acc = np.mean([nb_predict(nb, t) == c for t, c in test])
print(f"\naccuracy на тесте: {acc:.2f}")

**Почему модель так решила?** Для двух классов удобно смотреть на разность логарифмов: вклад слова $w$ равен $\log P(w\mid +)-\log P(w\mid -)$. Положительный вклад тянет к «+», отрицательный к «−». Сумма вкладов плюс разность априорных логарифмов — это логарифм отношения шансов, апостериорная вероятность $P(+\mid d)=\sigma(\text{сумма})$.

In [ ]:
def explain(model, text, plot=False):
    s = nb_scores(model, text)
    prior = model["log_prior"]["+"] - model["log_prior"]["-"]
    rows = []
    for w in model["featurize"](text):
        if w in model["vocab"]:
            rows.append((w, model["log_lik"]["+"][w] - model["log_lik"]["-"][w]))
        else:
            rows.append((w, None))
    total = prior + sum(v for _, v in rows if v is not None)
    p_pos = 1 / (1 + np.exp(-total))
    print(text)
    for w, v in rows:
        print(f"   {w:<16}", "нет в словаре" if v is None else f"{v:+.2f}")
    print(f"   {'априорн.':<16} {prior:+.2f}\n   {'сумма':<16} {total:+.2f}  →  P(+|d) = {p_pos:.3f}  →  {max(s, key=s.get)}")
    if plot:
        known = [(w, v) for w, v in rows if v is not None]
        plt.figure(figsize=(7, 2.8))
        plt.barh([w for w, _ in known][::-1], [v for _, v in known][::-1],
                 color=["#17907E" if v > 0 else "#D9532C" for _, v in known][::-1])
        plt.axvline(0, color="black", lw=0.8)
        plt.xlabel("log P(w|+) − log P(w|−)")
        plt.title("Вклады слов: зелёные тянут к «+», красные к «−»")
        plt.tight_layout()
        plt.show()

explain(nb, "Фильм совсем не понравился, очень скучно", plot=True)
print()
explain(nb, "Совсем не скучно, актёры играют хорошо")

Сверим с scikit-learn: `MultinomialNB(alpha=1.0)` хранит те же величины в `class_log_prior_` и `feature_log_prob_`.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

cv = CountVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None)
Xtr = cv.fit_transform(X_train)
sk_nb = MultinomialNB(alpha=1.0).fit(Xtr, y_train)

vocab_sk = cv.get_feature_names_out()
ours = np.array([[nb["log_lik"][c][w] for w in vocab_sk] for c in sk_nb.classes_])
print("классы:", sk_nb.classes_)
print("априорные совпадают:    ", np.allclose(sk_nb.class_log_prior_, [nb["log_prior"][c] for c in sk_nb.classes_]))
print("правдоподобия совпадают:", np.allclose(sk_nb.feature_log_prob_, ours))
pred_sk = sk_nb.predict(cv.transform(X_test))
pred_ours = np.array([nb_predict(nb, t) for t in X_test])
print("предсказания на тесте совпадают:", np.array_equal(pred_sk, pred_ours))
for t, c, p in zip(X_test, y_test, pred_sk):
    print(f"   {'✓' if c == p else '✗'} истина {c} модель {p}  {t}")

## 3. Отрицание

Посмотрите на вклады выше. «Понравился» почти ничего не весит (+0,03): в обучении оно встречается и в «фильм понравился», и в «фильм не понравился», мешок слов не видит, что во втором случае перед ним стоит «не». Всю работу делает сама частица «не» (−1,00), потому что в отрицательных отзывах она встречается в три раза чаще. Из-за этого второй отзыв «Совсем не скучно, актёры играют хорошо» модель считает отрицательным: «не» тянет вниз, а «скучно» тоже встречалось в обоих классах.

Простой и известный приём (Das & Chen, 2001; Pang et al., 2002): приклеить отрицание к следующему слову. Тогда «не понравился» превращается в новый признак `НЕ_понравился`, «не скучно» — в `НЕ_скучно`, и у каждого из них своя статистика.

In [ ]:
NEG_WORDS = {"не", "ни", "нет"}

def mark_negation(text):
    # «не понравился» → «НЕ_понравился»: отрицание приклеивается к следующему слову
    out, neg = [], False
    for w in tokenize(text):
        if w in NEG_WORDS:
            neg = True
            continue
        out.append("НЕ_" + w if neg else w)
        neg = False
    return out

for t in ["Фильм совсем не понравился, очень скучно"] + [t for t in pos + neg if "понравился" in tokenize(t)][:2]:
    print(f"{t}\n   → {mark_negation(t)}")

nb_neg = train_nb(X_train, y_train, featurize=mark_negation)
print("\nновых признаков с отрицанием в словаре:", sorted(w for w in nb_neg["vocab"] if "_" in w))

tricky = [
    ("Фильм не понравился", "-"),
    ("Совсем не скучно и очень смешно", "+"),
    ("Не советую этот телефон", "-"),
    ("Ни разу не пожалел", "+"),
    ("Обслуживание не понравилось", "-"),
]
print(f"\n{'':<36} NB     NB+отриц.")
for t, gold in tricky:
    print(f"{t:<36} {nb_predict(nb, t):<6} {nb_predict(nb_neg, t):<6} (верно: {gold})")
acc_neg = np.mean([nb_predict(nb_neg, t) == c for t, c in test])
print(f"\naccuracy на тесте: NB {acc:.2f},  NB+отриц. {acc_neg:.2f}")
print()
explain(nb_neg, "Фильм совсем не понравился, очень скучно")

## 4. Линейный SVM: градиентный спуск по hinge loss

SVM тоже линейная модель: $\text{score}(d)=w\cdot x_d+b$, класс — знак score. Но веса подбираются иначе: метка $y\in\{+1,-1\}$, и мы хотим, чтобы каждый пример лежал не просто по нужную сторону, а с **отступом** не меньше 1: $y(w\cdot x+b)\ge 1$. Функция потерь

$$L(w,b)=\frac{\lambda}{2}\|w\|^2+\frac1N\sum_{i}\max\big(0,\;1-y_i(w\cdot x_i+b)\big).$$

Примеры с отступом больше 1 не влияют на градиент (они «далеко»), учатся только на нарушителях. Признаки — TF-IDF с отрицаниями. Учим полным градиентным спуском: каждая эпоха — один шаг по всем 48 обучающим отзывам.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(tokenizer=mark_negation, lowercase=False, token_pattern=None)
Xtr_tf = tf.fit_transform(X_train).toarray()
Xte_tf = tf.transform(X_test).toarray()
ytr = np.where(y_train == "+", 1, -1)
yte = np.where(y_test == "+", 1, -1)

def train_svm(X, y, lam=0.01, lr=0.5, epochs=300):
    N, D = X.shape
    w, b = np.zeros(D), 0.0
    hist = []
    for ep in range(epochs):
        margins = y * (X @ w + b)
        viol = margins < 1                                   # нарушители отступа
        loss = lam / 2 * w @ w + np.maximum(0, 1 - margins).mean()
        hist.append((loss, (margins > 0).mean(), viol.sum()))
        grad_w = lam * w - (y[viol, None] * X[viol]).sum(axis=0) / N
        grad_b = -y[viol].sum() / N
        w -= lr * grad_w
        b -= lr * grad_b
    return w, b, np.array(hist)

w_svm, b_svm, hist = train_svm(Xtr_tf, ytr)
for ep in (0, 10, 50, 100, 299):
    print(f"эпоха {ep:3d}: потери {hist[ep, 0]:.3f}, accuracy на обучении {hist[ep, 1]:.2f}, нарушителей отступа {int(hist[ep, 2])}")
acc_svm = np.mean(np.sign(Xte_tf @ w_svm + b_svm) == yte)
print(f"\naccuracy на тесте (наш SVM): {acc_svm:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(hist[:, 0], color="#2B55C9")
ax[0].set_title("Потери (hinge + L2)")
ax[0].set_xlabel("эпоха")
ax[1].plot(hist[:, 1], color="#17907E", label="accuracy на обучении")
ax[1].plot(hist[:, 2] / len(ytr), color="#D9532C", label="доля нарушителей отступа")
ax[1].set_xlabel("эпоха")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

Сравним с библиотечными моделями: `LinearSVC(loss="hinge")` решает ту же задачу (параметр $C$ соответствует $1/(\lambda N)$), `LogisticRegression` — та же линейная модель, но с логистической функцией потерь. Посмотрим на самые сильные слова: у SVM это веса $w$, у NB — разности логарифмов.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

N = Xtr_tf.shape[0]
svc = LinearSVC(loss="hinge", C=1 / (0.01 * N), max_iter=20000, random_state=0).fit(Xtr_tf, y_train)
logreg = LogisticRegression(C=10, max_iter=1000).fit(Xtr_tf, y_train)
w_sk = svc.coef_[0] if svc.classes_[1] == "+" else -svc.coef_[0]
print("корреляция весов: наш SVM и LinearSVC:", round(float(np.corrcoef(w_svm, w_sk)[0, 1]), 3))
for name, pred in [("наш SVM", np.where(Xte_tf @ w_svm + b_svm > 0, "+", "-")),
                   ("LinearSVC", svc.predict(Xte_tf)), ("LogisticRegression", logreg.predict(Xte_tf))]:
    print(f"   {name:<20} accuracy на тесте {np.mean(pred == y_test):.2f}")

vocab_tf = tf.get_feature_names_out()
order = np.argsort(w_svm)
nb_ratio = {w: nb_neg["log_lik"]["+"][w] - nb_neg["log_lik"]["-"][w] for w in nb_neg["vocab"]}
top_nb = sorted(nb_ratio, key=nb_ratio.get)
print("\nSVM  «+»:", ", ".join(vocab_tf[order[::-1][:8]]))
print("SVM  «−»:", ", ".join(vocab_tf[order[:8]]))
print("NB   «+»:", ", ".join(top_nb[::-1][:8]))
print("NB   «−»:", ", ".join(top_nb[:8]))

top = np.concatenate([order[:8], order[-8:]])
plt.figure(figsize=(7, 4.6))
plt.barh(vocab_tf[top], w_svm[top], color=["#D9532C"] * 8 + ["#17907E"] * 8)
plt.axvline(0, color="black", lw=0.8)
plt.title("Самые сильные веса SVM")
plt.tight_layout()
plt.show()

## 5. Оценка качества

На 12 тестовых отзывах сравним модели по **матрице ошибок** и метрикам: *точность* (precision) — какая доля отзывов, названных положительными, действительно положительные; *полнота* (recall) — какую долю положительных отзывов модель нашла; F1 — их среднее гармоническое.

У NB без отрицания на тесте есть ошибки, поэтому подробный отчёт выведем для него. 12 отзывов — очень мало, одна ошибка меняет accuracy на 8 процентных пунктов. Надёжнее **кросс-валидация**: разбиваем все 60 отзывов на 5 частей, 5 раз учим на четырёх и проверяем на пятой. Чтобы честно сравнить, словарь и IDF строятся заново внутри каждого разбиения (`Pipeline`).

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

models = {
    "NB": pred_ours,
    "NB+отриц.": np.array([nb_predict(nb_neg, t) for t in X_test]),
    "SVM": svc.predict(Xte_tf),
    "LogReg": logreg.predict(Xte_tf),
}
labels = ["+", "-"]
fig, axes = plt.subplots(1, len(models), figsize=(12, 3))
for ax, (name, pred) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, pred, labels=labels)
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=6)
    for r in range(2):
        for c in range(2):
            ax.text(c, r, cm[r, c], ha="center", va="center", fontsize=14, color="white" if cm[r, c] > 3 else "black")
    ax.set_xticks([0, 1], labels)
    ax.set_yticks([0, 1], labels)
    ax.set_xlabel("предсказано")
    ax.set_title(f"{name}: {np.mean(pred == y_test):.2f}")
axes[0].set_ylabel("на самом деле")
plt.tight_layout()
plt.show()

print("NB без отрицания:")
print(classification_report(y_test, models["NB"], digits=2, zero_division=0))

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

X_all = pos + neg
y_all = np.array(["+"] * len(pos) + ["-"] * len(neg))

def vec(neg_marking, tfidf):
    V = TfidfVectorizer if tfidf else CountVectorizer
    return V(tokenizer=mark_negation if neg_marking else tokenize, lowercase=False, token_pattern=None)

pipelines = {
    "NB": make_pipeline(vec(False, False), MultinomialNB()),
    "NB+отриц.": make_pipeline(vec(True, False), MultinomialNB()),
    "SVM": make_pipeline(vec(False, True), LinearSVC(C=1.0, random_state=0)),
    "SVM+отриц.": make_pipeline(vec(True, True), LinearSVC(C=1.0, random_state=0)),
    "LogReg+отриц.": make_pipeline(vec(True, True), LogisticRegression(C=10, max_iter=1000)),
}
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
res = {}
for name, pipe in pipelines.items():
    sc = cross_val_score(pipe, X_all, y_all, cv=cv5)
    res[name] = sc
    print(f"{name:<18} {sc.mean():.2f} ± {sc.std():.2f}   {np.round(sc, 2)}")

plt.figure(figsize=(8, 3.2))
plt.bar(list(res), [s.mean() for s in res.values()], yerr=[s.std() for s in res.values()],
        color="#2B55C9", capsize=4)
plt.ylim(0, 1.05)
plt.ylabel("accuracy")
plt.title("Кросс-валидация, 5 разбиений (среднее ± ст. откл.)")
plt.tight_layout()
plt.show()

Отрицание даёт прирост около 17 процентных пунктов и для NB, и для SVM, причём почти в каждом разбиении, а не только в среднем. А разница между NB и SVM с одинаковыми признаками (3 пункта) меньше стандартного отклонения: на 60 отзывах это шум. На таком маленьком корпусе признаки важнее выбора классификатора; для настоящих выводов нужны сотни и тысячи размеченных отзывов.

## 6. Задания

1. **Сглаживание.** Обучите `train_nb` с $\alpha = 0{,}01$, $0{,}1$, $1$, $5$ и выведите accuracy на тесте и вклад слова «понравился». Почему при маленьком $\alpha$ вклады редких слов становятся огромными?
2. **Область отрицания.** Измените `mark_negation` так, чтобы метка `НЕ_` ставилась на все слова до конца фразы (до запятой или точки; подсказка: сначала разбейте текст `re.split(r"[,.!?]", text)`, потом токенизируйте каждую часть). Улучшается ли кросс-валидация NB? Почему для коротких отзывов это может и не помочь?
3. **Свои отзывы.** Напишите 5 отзывов, на которых модель NB+отрицание ошибается (например, с сарказмом «Ну да, отличный фильм, уснул через 10 минут»). Какие слова дают ошибочный вклад? Можно ли их исправить, добавив 2–3 отзыва в обучение?
4. **Биграммы.** Замените `TfidfVectorizer(...)` в разделе 5 на вариант с `ngram_range=(1, 2)` (без `mark_negation`). Сравните кросс-валидацию SVM с отрицанием через `НЕ_` и с биграммами: какой способ лучше ловит «не понравился»?
5. **Нейтральный класс.** Добавьте 10 нейтральных отзывов (метка `"0"`, например «Посмотрел фильм в субботу») и обучите `MultinomialNB` и `LinearSVC` на трёх классах. Постройте матрицу ошибок 3 × 3. Какие классы путаются чаще?

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь